# UTS Project Data Mining — Analisis Sentimen Ulasan Indie Game Steam

**Eksperimen mandiri dengan tiga algoritma yang berbeda dari jurnal acuan:**
1. Logistic Regression
2. Linear Support Vector Machine (Linear SVM)
3. Random Forest

Jurnal acuan menggunakan **Naive Bayes** untuk klasifikasi sentimen dan **Latent Dirichlet Allocation (LDA)** untuk pemodelan topik. Notebook ini tidak mengulang kedua metode tersebut.

**Dataset:** 3.621 ulasan dari 24 game indie lokal di Steam. Label target berasal dari kolom `recommend` (`Recommended` / `Not Recommended`).

> Catatan metodologis: hasil pada notebook ini harus dihitung dari awal. Jangan mengisi laporan dengan angka hasil sebelum seluruh sel dijalankan.

## 1. Persiapan lingkungan

Jalankan sel berikut. Library `langdetect` digunakan untuk menyaring ulasan berbahasa Inggris, sejalan dengan kriteria seleksi bahasa pada jurnal. Hasil deteksi bahasa otomatis dapat berbeda sedikit dari proses penelitian asli.

In [ ]:
!pip -q install langdetect

## 2. Unggah dataset CSV

In [ ]:
from google.colab import files
uploaded = files.upload()

# Ambil nama file CSV yang diunggah
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Silakan unggah file dataset berformat CSV.")
csv_path = csv_files[0]
print("File:", csv_path)

## 3. Import library dan membaca dataset

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

DetectorFactory.seed = 42

# File sumber memiliki karakter Windows-1252 pada beberapa baris.
# Jika encoding ini gagal, coba latin-1.
try:
    df = pd.read_csv(csv_path, encoding="cp1252")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, encoding="latin-1")

print("Ukuran awal:", df.shape)
print("Kolom:", df.columns.tolist())
display(df.head())

## 4. Pembersihan struktur dan pemeriksaan data

In [ ]:
# Hilangkan kolom kosong tambahan dari hasil web scraping.
df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed")]
df.columns = df.columns.astype(str).str.strip()

required_columns = ["reviewtext", "recommend"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

# Sisakan kolom yang relevan untuk eksperimen.
data = df[["reviewtext", "recommend"]].copy()
data = data.dropna(subset=["reviewtext", "recommend"])

data["reviewtext"] = data["reviewtext"].astype(str).str.strip()
data["recommend"] = data["recommend"].astype(str).str.strip()

# Hapus baris dengan teks kosong dan label di luar dua kelas yang diharapkan.
data = data[data["reviewtext"].ne("")]
data = data[data["recommend"].isin(["Recommended", "Not Recommended"])]

# Hilangkan duplikasi berdasarkan teks dan label.
before_duplicates = len(data)
data = data.drop_duplicates(subset=["reviewtext", "recommend"]).reset_index(drop=True)

print("Jumlah setelah pembersihan dasar:", len(data))
print("Duplikat dihapus:", before_duplicates - len(data))
print("\nDistribusi label:")
display(data["recommend"].value_counts().to_frame("jumlah"))

## 5. Membersihkan teks dan menyaring bahasa Inggris

In [ ]:
def clean_review(text):
    text = str(text)
    # Hilangkan awalan tanggal hasil scraping, misalnya "Posted: October 17, 2020"
    text = re.sub(r"(?i)^\\s*Posted:\\s*[^\\n]*\\n?", " ", text)
    # Hilangkan URL
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)
    # Hilangkan tag HTML jika ada
    text = re.sub(r"<[^>]+>", " ", text)
    # Rapikan whitespace dan ubah ke huruf kecil
    text = re.sub(r"\\s+", " ", text).strip().lower()
    return text

data["clean_text"] = data["reviewtext"].apply(clean_review)
data = data[data["clean_text"].str.len() > 2].copy()

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

data["language"] = data["clean_text"].apply(is_english)
data = data[data["language"]].copy().reset_index(drop=True)

print("Jumlah ulasan setelah penyaringan bahasa Inggris:", len(data))
print("\nDistribusi label setelah penyaringan:")
display(data["recommend"].value_counts().to_frame("jumlah"))
display(data[["clean_text", "recommend"]].head())

## 6. Eksplorasi data

Periksa distribusi kelas sebelum pemodelan. Dataset bisa tidak seimbang; karena itu evaluasi tidak hanya menggunakan accuracy, tetapi juga precision, recall, dan F1-score.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="recommend", order=["Recommended", "Not Recommended"])
plt.title("Distribusi Label Rekomendasi")
plt.xlabel("Label")
plt.ylabel("Jumlah ulasan")
plt.xticks(rotation=10)
plt.show()

print("Jumlah data:", len(data))
print("Jumlah kelas:", data["recommend"].nunique())
print("Rasio kelas:")
display((data["recommend"].value_counts(normalize=True) * 100).round(2).to_frame("persentase"))

## 7. Menyiapkan data latih dan data uji

Label `Recommended` dipetakan menjadi **positive**, sedangkan `Not Recommended` menjadi **negative**. Data dibagi secara stratified dengan rasio 80:20 agar proporsi kelas tetap terjaga. Pembagian dilakukan sebelum TF-IDF dipelajari untuk menghindari data leakage.

In [ ]:
X = data["clean_text"]
y = data["recommend"].map({
    "Recommended": "positive",
    "Not Recommended": "negative"
})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data latih:", len(X_train))
print("Data uji:", len(X_test))
print("\nDistribusi label data latih:")
display(y_train.value_counts().to_frame("jumlah"))
print("\nDistribusi label data uji:")
display(y_test.value_counts().to_frame("jumlah"))

## 8. Membangun tiga model klasifikasi

Semua model menggunakan representasi teks TF-IDF dan pembagian data yang sama. Parameter `class_weight='balanced'` pada Logistic Regression dan Linear SVM membantu mengurangi bias terhadap kelas mayoritas. Random Forest juga menggunakan bobot kelas seimbang.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Linear SVM": LinearSVC(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

pipelines = {}
for name, model in models.items():
    pipelines[name] = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            stop_words="english"
        )),
        ("classifier", model)
    ])

print("Model siap:", list(pipelines.keys()))

## 9. Melatih dan mengevaluasi model

In [ ]:
results = []
predictions = {}

for name, pipeline in pipelines.items():
    print(f"Melatih {name} ...")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (macro)", ascending=False)
display(results_df.round(4))

## 10. Classification report dan confusion matrix

In [ ]:
for name, y_pred in predictions.items():
    print("=" * 70)
    print(name)
    print(classification_report(y_test, y_pred, labels=["negative", "positive"], zero_division=0))

    cm = confusion_matrix(y_test, y_pred, labels=["negative", "positive"])
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["negative", "positive"],
        yticklabels=["negative", "positive"]
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Prediksi")
    plt.ylabel("Aktual")
    plt.show()

## 11. Visualisasi perbandingan performa

In [ ]:
metrics_to_plot = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"]
plot_df = results_df.set_index("Model")[metrics_to_plot]

ax = plot_df.plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
ax.set_title("Perbandingan Performa Model")
ax.set_ylabel("Skor")
ax.set_xlabel("Algoritma")
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Analisis kesalahan prediksi

Bagian ini mengambil contoh ulasan yang salah diprediksi oleh model terbaik berdasarkan F1-score macro. Baca contoh tersebut untuk melihat apakah kesalahan terjadi karena sarkasme, kalimat campuran, konteks game, atau kata-kata yang ambigu.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = pipelines[best_model_name]
best_pred = predictions[best_model_name]

error_analysis = pd.DataFrame({
    "review": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(best_pred)
})
errors = error_analysis[error_analysis["actual"] != error_analysis["predicted"]]

print("Model terbaik berdasarkan F1 macro:", best_model_name)
print("Jumlah prediksi salah:", len(errors))
display(errors.head(20))

## 13. Uji prediksi ulasan baru

In [ ]:
sample_reviews = [
    "This game has an amazing story and beautiful music. I really love it.",
    "The game is full of bugs and the controls are terrible.",
    "It is okay, but the gameplay could be better."
]

sample_predictions = best_pipeline.predict(sample_reviews)
for review, label in zip(sample_reviews, sample_predictions):
    print(f"Ulasan: {review}")
    print(f"Prediksi sentimen: {label}")
    print("-" * 60)

## 14. Simpan hasil eksperimen

File hasil metrik dan prediksi dapat diunduh untuk dipakai pada BAB IV laporan.

In [ ]:
results_df.to_csv("hasil_perbandingan_model.csv", index=False)
error_analysis.to_csv("hasil_prediksi_dan_kesalahan.csv", index=False)

from google.colab import files
files.download("hasil_perbandingan_model.csv")
files.download("hasil_prediksi_dan_kesalahan.csv")

## 15. Panduan interpretasi untuk laporan

Setelah notebook selesai dijalankan, catat:
- Jumlah data awal dan jumlah data setelah pembersihan.
- Jumlah data berbahasa Inggris yang tersisa.
- Distribusi kelas positive dan negative.
- Accuracy, macro-precision, macro-recall, dan macro-F1 setiap model.
- Confusion matrix masing-masing model.
- Model terbaik berdasarkan **macro-F1**, bukan accuracy saja.
- Contoh ulasan yang salah diklasifikasikan dan kemungkinan penyebabnya.

**Penting:** dataset sumber berisi 3.621 baris. Jurnal melaporkan 3.621 ulasan awal, lalu menyaring ulasan bahasa Inggris dan melakukan proses penyeimbangan data tersendiri. Notebook ini menerapkan split stratified 80:20 pada data bahasa Inggris yang lolos preprocessing, sehingga hasilnya tidak dimaksudkan untuk mereplikasi angka akurasi jurnal. Metode yang diuji juga berbeda dari Naive Bayes dan LDA pada penelitian acuan.